# Credit review agent: from model score to a memo for the underwriter

## Problem
A credit model gives a score, but an underwriter still has to check many things before deciding: why the score is high, whether any policy rule is broken, how similar loans performed in the past, and what to ask the applicant. This takes time for every application.

## Approach
An LLM **agent** that prepares the review, and a **human makes the decision**.
- The agent has **tools**. It decides which tools to call and in what order.
- Every number in the memo comes from a tool. The LLM only **collects, reads and summarises**.
- Tools use the EBM credit model from `01_credit_scoring_ebm.ipynb`.

| Tool | What it returns |
|---|---|
| `get_application` | The application data |
| `score_application` | EBM default probability and risk decile |
| `get_reason_codes` | Features that push the risk up or down (exact EBM contributions) |
| `check_policy_rules` | Simple credit policy rules that are broken |
| `find_similar_cases` | Default rate of similar past loans (2010–2013) |

Run `01_credit_scoring_ebm.ipynb` first: it saves `model/credit_models.joblib` and `model/modeling_data.pkl`.

## 1. Setup

In [1]:
import json
import re
import time

import joblib
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_colwidth", 200)
load_dotenv("../.env")

CHAT_MODEL = "gpt-4o-mini"
RANDOM_STATE = 42

art = joblib.load("../model/credit_models.joblib")
ebm, FEATURES = art["ebm"], art["features"]
NUM_FEATURES, CAT_FEATURES = art["num_features"], art["cat_features"]

data = pd.read_pickle("../model/modeling_data.pkl")
past = data[data["issue_year"].isin(art["train_years"])]          # outcomes known: used for similar cases
new = data[data["issue_year"] == art["test_year"]]                # "new applications" to review
print(f"past loans {len(past):,}   new applications {len(new):,}")

past loans 167,149   new applications 283,026


## 2. Tools
Normal Python functions with the `@tool` decorator. The docstring tells the LLM what the tool does and when to use it, so it matters.

The tools never return the real outcome (`default`) of a new application.

In [2]:
# --- prepared once, used by the tools
# predict probabilities and deciles for all new applications
p_new = pd.Series(ebm.predict_proba(new[FEATURES])[:, 1], index=new.index)
decile = pd.Series(pd.qcut(p_new.rank(method="first"), 10, labels=range(1, 11)).astype(int), index=new.index)

# --- policy rules for illustrative purposes (not Lending Club's real policy)
POLICY = {                      # illustrative rules, not Lending Club's real policy
    "dti_max": 35,              # debt-to-income (%)
    "fico_min": 660,
    "inq_6m_max": 3,            # credit inquiries in the last 6 months
    "loan_to_income_max": 0.40,
    "revol_util_max": 90,       # credit card utilisation (%)
    "unverified_loan_max": 20000,
}

# --- find similar past loans (2010-2013) for each new application
SIM_COLS = ["fico", "dti", "loan_to_income", "annual_inc", "inq_last_6mths", "revol_util"]
past_s = past.sample(min(len(past), 100_000), random_state=RANDOM_STATE)
past_sim = past_s[SIM_COLS].fillna(past_s[SIM_COLS].median())
past_sim = past_sim.assign(annual_inc=np.log1p(past_sim["annual_inc"]))
scaler = StandardScaler().fit(past_sim)
knn = NearestNeighbors(n_neighbors=200).fit(scaler.transform(past_sim))


# --- helper to get one row of the new applications data
def _row(application_id):
    if application_id not in new.index:
        raise ValueError(f"Unknown application_id {application_id}")
    return new.loc[[application_id], FEATURES]

# --- helper to round numeric values for display
def _num(v):
    return None if pd.isna(v) else round(float(v), 3)


# --- tools for the LLM to call
# 普通のPython関数に @tool を付けただけで、LLMが呼び出せる道具になります。関数の引数は、LLMが理解できるように型を付けてください。返り値は、LLMが理解できるように dict にしてください。
# 関数の説明文（docstring）です。 LLMはこの説明を読んで、「この道具は何をするのか」「いつ使うべきか」を判断します。人間向けのコメントではなく、LLMへの使い方の説明書

# 1. 申込の中身（借入額、収入、FICOなど）を返す
@tool
def get_application(application_id: int) -> dict:
    """Return the application data (loan, income, credit history) for one application."""
    r = _row(application_id).iloc[0]
    return {f: (_num(r[f]) if f in NUM_FEATURES else r[f]) for f in FEATURES}

# 2.  先に計算したデフォルト確率と順位（1〜10）、その年の平均を返す
@tool
def score_application(application_id: int) -> dict:
    """Score one application with the EBM credit model. Returns the probability of default
    and the risk decile among this year's applications (1 = safest 10%, 10 = riskiest 10%)."""
    _row(application_id)
    return {"probability_of_default": round(float(p_new[application_id]), 4),
            "risk_decile": int(decile[application_id]),
            "average_probability_this_year": round(float(p_new.mean()), 4)}

# 3. 特徴量の寄与度を説明する
FEATURE_LABELS = {
    "loan_amnt": "loan amount ($)",
    "annual_inc": "annual income ($)",
    "loan_to_income": "loan amount / annual income",
    "dti": "debt-to-income ratio (%)",
    "fico": "FICO score",
    "emp_length_yrs": "years of employment",
    "credit_hist_months": "length of credit history (months)",
    "delinq_2yrs": "late payments in the last 2 years",
    "inq_last_6mths": "credit inquiries in the last 6 months",
    "open_acc": "number of open credit lines",
    "pub_rec": "number of negative public records",
    "revol_bal": "credit card balance ($)",
    "revol_util": "credit card utilisation (%)",
    "total_acc": "total number of credit lines",
    "mort_acc": "number of mortgage accounts",
    "pub_rec_bankruptcies": "number of past bankruptcies",
    "home_ownership": "home ownership",
    "verification_status": "income verification",
    "purpose": "loan purpose",
    "application_type": "application type",
}

def _reason_text(term, X, c):
    parts = []
    for f in term.split(" & "):
        v = X.iloc[0][f]
        if pd.isna(v):
            v = "missing"
        elif f in NUM_FEATURES:
            v = f"{v:,.0f}" if abs(v) >= 100 else f"{v:.2f}"
        parts.append(f"{FEATURE_LABELS.get(f, f)} is {v}")
    strength = "strongly " if abs(c) >= 0.3 else ("slightly " if abs(c) < 0.1 else "")
    direction = "increases" if c > 0 else "decreases"
    prefix = "The combination: " if " & " in term else ""
    return f"{prefix}{' and '.join(parts)}. This {strength}{direction} the risk."


@tool
def get_reason_codes(application_id: int, top_n: int = 4) -> dict:
    """Explain the score of one application. Returns the features that increase the risk most and
    the features that decrease it most, as plain-English reasons with exact EBM contributions (log-odds)."""
    X = _row(application_id)
    contrib = pd.Series(ebm.eval_terms(X)[0], index=ebm.term_names_)

    def describe(term):
        c = float(contrib[term])
        return {"reason": _reason_text(term, X, c), "contribution": round(c, 3)}

    up = contrib[contrib > 0].sort_values(ascending=False).head(top_n)
    down = contrib[contrib < 0].sort_values().head(top_n)
    return {"increase_risk": [describe(t) for t in up.index],
            "decrease_risk": [describe(t) for t in down.index]}


# 4. 7つのルールを1つずつ確かめて、引っかかったものだけを返す
@tool
def check_policy_rules(application_id: int) -> dict:
    """Check one application against the credit policy rules. Returns the rules that are broken."""
    r = _row(application_id).iloc[0]
    checks = [
        ("DTI above maximum", r["dti"] > POLICY["dti_max"], f"dti {_num(r['dti'])}% > {POLICY['dti_max']}%"),
        ("FICO below minimum", r["fico"] < POLICY["fico_min"], f"fico {_num(r['fico'])} < {POLICY['fico_min']}"),
        ("Many recent credit inquiries", r["inq_last_6mths"] > POLICY["inq_6m_max"],
         f"{_num(r['inq_last_6mths'])} inquiries in 6 months > {POLICY['inq_6m_max']}"),
        ("Loan large compared with income", r["loan_to_income"] > POLICY["loan_to_income_max"],
         f"loan_to_income {_num(r['loan_to_income'])} > {POLICY['loan_to_income_max']}"),
        ("High credit card utilisation", r["revol_util"] > POLICY["revol_util_max"],
         f"revol_util {_num(r['revol_util'])}% > {POLICY['revol_util_max']}%"),
        ("Past bankruptcy", r["pub_rec_bankruptcies"] > 0, f"{_num(r['pub_rec_bankruptcies'])} bankruptcies"),
        ("Large loan with unverified income",
         (r["verification_status"] == "Not Verified") and r["loan_amnt"] > POLICY["unverified_loan_max"],
         f"loan {_num(r['loan_amnt'])} with income not verified"),
    ]
    broken = [{"rule": name, "detail": detail} for name, hit, detail in checks if bool(hit)]
    return {"rules_broken": broken, "n_rules_checked": len(checks)}


# 5. 似た過去の融資200件を探して、そのデフォルト率と、過去全体の平均を返す
@tool
def find_similar_cases(application_id: int) -> dict:
    """Find 200 past loans (2010-2013) with a similar profile (FICO, DTI, loan to income, income,
    inquiries, utilisation) and return how often they defaulted, compared with all past loans."""
    r = _row(application_id)[SIM_COLS].fillna(past_s[SIM_COLS].median())
    r = r.assign(annual_inc=np.log1p(r["annual_inc"]))
    idx = knn.kneighbors(scaler.transform(r), return_distance=False)[0]
    return {"n_similar_loans": len(idx),
            "default_rate_similar": round(float(past_s["default"].iloc[idx].mean()), 4),
            "default_rate_all_past": round(float(past_s["default"].mean()), 4)}


TOOLS = [get_application, score_application, get_reason_codes, check_policy_rules, find_similar_cases]

# try the tools directly (no LLM)
example_id = int(p_new.idxmax())
print(json.dumps(score_application.invoke({"application_id": example_id}), indent=2))
print(json.dumps(check_policy_rules.invoke({"application_id": example_id}), indent=2))
print(json.dumps(get_reason_codes.invoke({"application_id": example_id}), indent=2))

{
  "probability_of_default": 0.6408,
  "risk_decile": 10,
  "average_probability_this_year": 0.1296
}
{
  "rules_broken": [
    {
      "rule": "Many recent credit inquiries",
      "detail": "5.0 inquiries in 6 months > 3"
    },
    {
      "rule": "Loan large compared with income",
      "detail": "loan_to_income 0.5 > 0.4"
    },
    {
      "rule": "Past bankruptcy",
      "detail": "1.0 bankruptcies"
    }
  ],
  "n_rules_checked": 7
}
{
  "increase_risk": [
    {
      "reason": "credit inquiries in the last 6 months is 5.00. This strongly increases the risk.",
      "contribution": 0.582
    },
    {
      "reason": "loan amount / annual income is 0.50. This strongly increases the risk.",
      "contribution": 0.582
    },
    {
      "reason": "FICO score is 662. This strongly increases the risk.",
      "contribution": 0.429
    },
    {
      "reason": "number of open credit lines is 33.00. This strongly increases the risk.",
      "contribution": 0.384
    }
  ],
  "decrea

## 3. The agent
`create_agent` connects the LLM and the tools. The LLM reads the request, calls a tool, reads the result, and decides the next step, until it can write the memo.

The system prompt sets the rules: use the tools for every fact, do not make the final decision, and use a fixed memo format.

In [3]:
SYSTEM_PROMPT = """You are a credit review assistant. You prepare a review memo for a human underwriter.
You do NOT make the final credit decision.

Rules:
- Use the tools to get every fact and number. Never guess or invent numbers.
- Call all the tools before writing the memo.
- Write in plain English. Explain features in business words (e.g. "debt-to-income", not "dti").

Memo format:
1. Summary: probability of default, risk decile, and a suggested action: Approve, Review, or Decline.
2. Main risk drivers (from the reason codes).
3. Mitigating factors.
4. Policy rules broken (or "None").
5. Similar past loans: their default rate vs the average.
6. Questions for the underwriter to check with the applicant.
"""

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
agent = create_agent(llm, tools=TOOLS, system_prompt=SYSTEM_PROMPT)


def review(application_id, show_steps=True):
    t0 = time.time()
    out = agent.invoke({"messages": [{"role": "user", "content": f"Review application {application_id}."}]})
    msgs = out["messages"]
    calls = [c["name"] for m in msgs for c in (getattr(m, "tool_calls", None) or [])]
    if show_steps:
        print(f"tools called ({len(calls)}): {' -> '.join(calls)}   time {time.time() - t0:.0f}s\n")
    return {"application_id": application_id, "memo": msgs[-1].content, "tools_called": calls,
            "tool_outputs": [m.content for m in msgs if m.type == "tool"]}

## 4. Try it on three applications
A low-risk, a borderline and a high-risk application (by EBM score).

In [4]:
order = p_new.sort_values().index
cases = {"low risk": int(order[len(order) // 20]),
         "borderline": int(order[int(len(order) * 0.85)]),
         "high risk": int(order[-1])}

results = {}
for label, app_id in cases.items():
    print("=" * 30, label, f"(application {app_id})", "=" * 30)
    results[label] = review(app_id)
    print(results[label]["memo"], "\n")

============================== low risk (application 156364) ==============================
tools called (5): get_application -> score_application -> get_reason_codes -> check_policy_rules -> find_similar_cases   time 6s

**Memo for Underwriter Review: Application 156364**

**Summary:**
- Probability of Default: 3.72%
- Risk Decile: 1 (safest 10%)
- Suggested Action: Approve

**Main Risk Drivers:**
1. The number of open credit lines is 14, which slightly increases the risk.
2. The debt-to-income ratio is 23.26%, which also slightly increases the risk.
3. The income verification status is "Source Verified," contributing slightly to the risk.
4. The combination of an annual income of $120,000 and 10 mortgage accounts slightly increases the risk.

**Mitigating Factors:**
1. The applicant has a strong annual income of $120,000, which significantly decreases the risk.
2. The FICO score is 722, indicating good creditworthiness and decreasing the risk.
3. The applicant has 10 mortgage account

In [5]:
# for reference only: what actually happened to these loans (the agent never sees this)
pd.DataFrame({"application_id": cases.values(),
              "probability_of_default": [round(p_new[i], 3) for i in cases.values()],
              "actual_default": [int(new.loc[i, "default"]) for i in cases.values()]}, index=cases.keys())

,application_id,probability_of_default,actual_default
low risk,156364,0.037,0
borderline,216930,0.200,0
high risk,183999,0.641,0


## 5. Evaluation
For an agent, a good-looking memo is not enough. I check three things on a sample of applications:
1. **Process**: did the agent call all five tools?
2. **Groundedness**: is every number and statement in the memo supported by the tool outputs? (LLM judge, same idea as in the RAG notebook)
3. **Consistency with the model**: does the suggested action follow the risk decile?

In [6]:
class Claim(BaseModel):
    claim: str
    supported: bool


class Judgement(BaseModel):
    claims: list[Claim]


judge_prompt = ChatPromptTemplate.from_messages([
    ("system", "You check a credit memo against the tool outputs it was based on. "
               "Split the memo into short factual claims (numbers, rules, drivers). "
               "For each claim, say if the tool outputs support it. Ignore questions and suggestions."),
    ("human", "Tool outputs:\n{tools}\n\nMemo:\n{memo}"),
])
judge = judge_prompt | ChatOpenAI(model=CHAT_MODEL, temperature=0).with_structured_output(Judgement)


def suggested_action(memo):
    m = re.search(r"suggested action\W*(approve|review|decline)", memo.lower())
    return m.group(1) if m else "unclear"


sample_ids = new.sample(10, random_state=RANDOM_STATE).index.tolist()
rows = []
for app_id in sample_ids:
    r = review(app_id, show_steps=False)
    claims = judge.invoke({"tools": "\n".join(r["tool_outputs"]), "memo": r["memo"]}).claims
    rows.append({"application_id": app_id,
                 "risk_decile": int(decile[app_id]),
                 "all_tools_called": set(r["tools_called"]) == {t.name for t in TOOLS},
                 "groundedness": np.mean([c.supported for c in claims]) if claims else np.nan,
                 "unsupported": [c.claim for c in claims if not c.supported],
                 "suggested_action": suggested_action(r["memo"])})
ev = pd.DataFrame(rows).sort_values("risk_decile")
ev

,application_id,risk_decile,all_tools_called,groundedness,unsupported,suggested_action
3,112636,2,True,1.000000,[],approve
4,137593,2,True,1.000000,[],approve
8,403593,3,True,1.000000,[],approve
0,319678,4,True,1.000000,[],review
5,306352,5,True,1.000000,[],review
7,165216,6,True,0.928571,[Loan amount is 18.2% of annual income],review
1,366040,6,True,1.000000,[],review
9,360235,9,True,1.000000,[],review
6,323465,10,True,1.000000,[],decline
2,103132,10,True,1.000000,[],decline


In [7]:
print(f"all tools called: {ev['all_tools_called'].mean():.0%}")
print(f"mean groundedness: {ev['groundedness'].mean():.2f}")
ev.groupby("suggested_action")["risk_decile"].agg(["count", "mean", "min", "max"])

all tools called: 100%
mean groundedness: 0.99


,count,mean,min,max
suggested_action,,,,
approve,3,2.333333,2,3
decline,2,10.000000,10,10
review,5,6.000000,4,9


## 6. Notes
Write after running:
- Did the agent always call all the tools? In which order?
- Were there unsupported claims in the memos? What kind?
- Does the suggested action follow the risk decile? Where does it disagree, and why?

Design choices:
- **Human in the loop**: the agent prepares the review; the underwriter decides.
- **Numbers come from tools, not from the LLM**: the LLM reads and summarises. This makes the memo checkable.
- **Audit trail**: the list of tool calls and outputs is kept for every review.
- The policy rules here are simple examples. A real bank would use its own credit policy.